In [ ]:
# import libraries
import os
import sys

import mlflow
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data.load_data import load_raw_data, encode_target
from src.data.clean_data import clean_pipeline
from src.features.build_features import build_features_pipeline
from src.models.train import run_all_models
from src.models.stacking import train_and_evaluate_stack
from src.models.preprocessing import fit_frequency_encoders, apply_frequency_encoders, HIGH_CARDINALITY_CATEGORICAL
from src.models.evaluate import find_optimal_threshold, get_confusion_matrix_breakdown
from src.business.roi_simulation import compare_f1_and_roi_thresholds

from src.models.preprocessing import all_model_features
from src.explainability.fairness_audit import subgroup_performance, flag_disparate_subgroups

In [ ]:
# load raw data
data = encode_target(load_raw_data('../data/raw/bank-full.csv'))
cleaned = clean_pipeline(data)
print("Data loaded and cleaned successfully.")

In [ ]:
train_data, temp_data = train_test_split(cleaned, test_size=0.2, random_state=42, stratify=cleaned['y'])  # train-test split
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, stratify=temp_data['y'])  # validation-test split
train_features, train_target = build_features_pipeline(train_data, {'val' : val_data, 'test' : test_data})

In [ ]:
# train and compare five models: Logistic Regression, Random Forest, Gradient Boosting, XGBoost, LightGBM
mlflow.set_experiment("Bank Marketing Classification")
comparison_df = run_all_models(train_features, train_target['test'])
comparison_df.sort_values('precision', ascending=False)

In [ ]:
# stacked model
freq_encoders = fit_frequency_encoders(train_features, HIGH_CARDINALITY_CATEGORICAL)
train_encoders = apply_frequency_encoders(train_features, freq_encoders)
test_encoders = apply_frequency_encoders(train_target['test'], freq_encoders)
stack_result = train_and_evaluate_stack(train_features, train_target['test'], train_encoders, test_encoders)
print({k: v for k, v in stack_result.items() if k != 'pipeline'})

In [ ]:
# Threshold tuning + confusion-matrix breakdown for the best model
from src.models.train import get_base_models, train_and_log_model

n_pos = (train_features['y'] == 1).sum()
n_neg = (train_features['y'] == 0).sum()
best_model_name = comparison_df.sort_values('precision', ascending=False).iloc[0]['model_name']
estimator, need_scaler = get_base_models(n_neg/n_pos)[best_model_name]

# Re-fit the winner once more so we have a fitted pipeline object to reuse
best_result = train_and_log_model(best_model_name, estimator, need_scaler, train_features, train_target['test'])
y_test = test_encoders['y']
print(f"Best model: {best_model_name}, test precision: {best_result['precision']:.4f}, test recall: {best_result['recall']:.4f}")


In [ ]:
# ROI analysis - F1 optimal vs ROI optimal threshold
y_proba = best_result['pipeline'].predict_proba(test_encoders[all_model_features])[:, 1]
thresholds_df = find_optimal_threshold(y_test, y_proba)
print("Thresholds for F1 and ROI optimization:")
print(thresholds_df)

roi_comparison_df = compare_f1_and_roi_thresholds(y_test.values, y_proba, thresholds_df['threshold'])
roi_comparison_df

In [ ]:
# Fairness audit across client subgroups
audit_df = test_encoders.copy()
audit_df['y_pred'] = (y_proba >= thresholds_df['threshold']).astype(int)
audit_df['y_proba'] = y_proba

subgroup_performance_df = subgroup_performance(audit_df, 'y', 'y_pred', 'y_proba', 'age_life_stage')
subgroup_performance_df

In [ ]:
flag_disparate_subgroups(subgroup_performance_df, metric='recall', threshold=0.1)